In [1]:
! pip install torchaudio=="2.8.0" --index-url https://download.pytorch.org/whl/cu128 -q

# Data

In [2]:
! wget -O Test_phase2.csv https://api.zindi.world/v1/competitions/google-waxal-asr-challenge/files/Test_Phase2.csv?auth_token=zus.v1.82SPxDb.fMAbymuPfyJpuTpGHT4WNYQLttmmDj

zsh:1: no matches found: https://api.zindi.world/v1/competitions/google-waxal-asr-challenge/files/Test_Phase2.csv?auth_token=zus.v1.82SPxDb.fMAbymuPfyJpuTpGHT4WNYQLttmmDj


In [3]:
! wget https://storage.googleapis.com/waxalphase2/newaudios.zip

--2026-08-06 03:39:43--  https://storage.googleapis.com/waxalphase2/newaudios.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 172.253.155.207, 74.125.201.207, 142.251.183.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|172.253.155.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1086719156 (1.0G) [application/zip]
Saving to: ‘newaudios.zip.2’

newaudios.zip.2     100%[===================>]   1.01G   138MB/s    in 7.9s    

2026-08-06 03:39:51 (132 MB/s) - ‘newaudios.zip.2’ saved [1086719156/1086719156]



In [4]:
! unzip newaudios.zip -d phase_2_data/

Archive:  newaudios.zip
replace phase_2_data/__MACOSX/._newaudios? [y]es, [n]o, [A]ll, [N]one, [r]ename: ^C


In [2]:
import pandas as pd

In [3]:
test_df = pd.read_csv("Test_Phase2.csv")

In [4]:
test_df

,ID,Target
0,ID_QNYPTX,the quick brown fox
1,ID_CLCVQW,the quick brown fox
2,ID_INJFRV,the quick brown fox
3,ID_XKFXQJ,the quick brown fox
4,ID_TYVXWN,the quick brown fox
...,...,...
887,ID_CUDWBK,the quick brown fox
888,ID_ARVDNP,the quick brown fox
889,ID_EROPVG,the quick brown fox
890,ID_VYRCXO,the quick brown fox


In [5]:
from datasets import Dataset, Audio

test_dataset = Dataset.from_pandas(test_df)

In [6]:
dir_path = "phase_2_data/newaudios/"

# Create a new column in the dataset with the full path to the audio files
test_dataset = test_dataset.map(lambda x: {"audio": dir_path + x["ID"] + ".wav"}, remove_columns=["ID"])

Map:   0%|          | 0/892 [00:00<?, ? examples/s]

In [7]:
# Cast to Audio type and set the sampling rate to 16kHz
test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16000))

In [8]:
test_dataset[0]['audio']["array"]

array([0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 2.9574825e-05,
       2.9720512e-05, 4.9052705e-05], dtype=float32)

In [9]:
test_audios = [item['audio']["array"] for item in test_dataset]
sample_rates = [item['audio']["sampling_rate"] for item in test_dataset]

# Model

In [10]:
from transformers import AutoModelForCTC, AutoProcessor
import torch, torchaudio

MODEL_NAME = "ayymen/w2v-bert-2.0-waxal_lin_sna-1epoch"

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model     = AutoModelForCTC.from_pretrained(MODEL_NAME)
if torch.cuda.is_available():
    model.to("cuda")
model.eval()

Wav2Vec2BertForCTC(
  (wav2vec2_bert): Wav2Vec2BertModel(
    (feature_projection): Wav2Vec2BertFeatureProjection(
      (layer_norm): LayerNorm((160,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=160, out_features=1024, bias=True)
      (dropout): Dropout(p=0.05, inplace=False)
    )
    (encoder): Wav2Vec2BertEncoder(
      (dropout): Dropout(p=0.05, inplace=False)
      (layers): ModuleList(
        (0-23): 24 x Wav2Vec2BertEncoderLayer(
          (ffn1_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (ffn1): Wav2Vec2BertFeedForward(
            (intermediate_dropout): Dropout(p=0.0, inplace=False)
            (intermediate_dense): Linear(in_features=1024, out_features=4096, bias=True)
            (intermediate_act_fn): SiLU()
            (output_dense): Linear(in_features=4096, out_features=1024, bias=True)
            (output_dropout): Dropout(p=0.05, inplace=False)
          )
          (self_attn_layer_norm): LayerNorm(

In [11]:
path = "phase_2_data/newaudios/ID_ISOCJR.wav"

In [12]:
# Display audio
from IPython.display import Audio
Audio(path)

In [13]:
waveform, sr = torchaudio.load(path)
if sr != 16_000:
    waveform = torchaudio.functional.resample(waveform, sr, 16_000)

inputs = processor(
    waveform.squeeze().numpy(), sampling_rate=16_000, return_tensors="pt"
)

# Move input tensors to the same device as the model (cuda)
if torch.cuda.is_available():
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

with torch.no_grad():
    logits = model(**inputs).logits          # (1, T, vocab)

pred_ids   = torch.argmax(logits, dim=-1)
transcript = processor.decode(pred_ids[0])
print(transcript)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorc

In [14]:
def asr_inference(audio, sr):
    if sr != 16_000:
        waveform = torchaudio.functional.resample(torch.tensor(audio), sr, 16_000)
    else:
        waveform = torch.tensor(audio)

    inputs = processor(
        waveform.squeeze().numpy(), sampling_rate=16_000, return_tensors="pt"
    )

    # Move input tensors to the same device as the model (cuda)
    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits          # (1, T, vocab)

    pred_ids   = torch.argmax(logits, dim=-1)
    transcript = processor.decode(pred_ids[0])
    return transcript

In [15]:
from tqdm import tqdm
predictions = []
for audio, sr in tqdm(zip(test_audios, sample_rates), total=len(test_audios)):
    pred = asr_inference(audio, sr)
    predictions.append(pred)
    # print(pred)

100%|██████████| 892/892 [05:56<00:00,  2.50it/s]


In [16]:
# Add the predictions to the test_df
test_df["Target"] = predictions

In [17]:
test_df

,ID,Target
0,ID_QNYPTX,Bandeko awa na misu na ngai nazomona obele ban...
1,ID_CLCVQW,Biloko nazomona awa namoni eza obele mama moko...
2,ID_INJFRV,Ishangu mbiri dzemativi ose emakumbo. Dzine ru...
3,ID_XKFXQJ,Motuka esali mikolo ebele ezali akufa liboso y...
4,ID_TYVXWN,Miti midiki yakati rebeyi ine matanda matete n...
...,...,...
887,ID_CUDWBK,Munhukadzi akachena. Akapfeka hembe ine ruvara...
888,ID_ARVDNP,Photo elakisi biso mwa loboko esimbi mwa kisi ...
889,ID_EROPVG,Miti yemijakaranda mihombe ine tirangi yakakur...
890,ID_VYRCXO,Awa eza esika bakonzi batelemelaka makambo ya ...


In [18]:
# Find empty transcripts
empty_transcripts = test_df[test_df["Target"] == ""]
empty_transcripts

,ID,Target
106,ID_ISOCJR,


In [19]:
# Replace empty transcripts with a space to produce a valid submission
test_df.loc[empty_transcripts.index, "Target"] = " "

In [20]:
test_df[test_df["Target"] == ""]

,ID,Target


In [21]:
test_df.to_csv("submission.csv", index=False)